# SQL Server → Python → Visualization Pipeline

**Author:** Khaylub Thompson-Calvin · Portfolio project from PCC CIS277A (Data Analytics) coursework

This notebook demonstrates an end-to-end data pipeline: query a SQL Server database with `pyodbc`, load results into a pandas DataFrame, reshape with `pivot`, and visualize with matplotlib.

**Data:** U.S. Social Security Administration baby-names dataset (public data), hosted on a course database server. The database is only reachable with course access, so this notebook is shown with its saved output; the code pattern is fully reusable against any SQL Server instance.

**Note on credentials:** connection settings are read from environment variables. Never hard-code server names, usernames, or passwords in a notebook.

In [ ]:
import os

import pyodbc
import pandas as pd
import matplotlib.pyplot as plt

# Connection settings come from environment variables (never hard-coded).
connection = pyodbc.connect(
    server=os.environ["DB_SERVER"],
    database=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    driver="{ODBC Driver 17 for SQL Server}",
)

## Query: two name spellings over time

Pull the yearly counts for the names *Marc* and *Mark* (male) so we can compare how the two spellings trended across a century.

In [ ]:
df = pd.read_sql(
    """
    SELECT Name, Gender, Year, NameCount
    FROM all_data
    WHERE (Name = 'Marc' OR Name = 'Mark')
      AND Gender = 'M'
    ORDER BY Year
    """,
    connection,
)
df.head()

## Reshape and plot

Pivot the long-format result (one row per name per year) into one column per name, indexed by year, then plot both series.

In [ ]:
name_data = df.pivot(index="Year", columns="Name", values="NameCount")
name_data.plot()
plt.ylabel("Babies named per year")
plt.title("Marc vs. Mark, U.S. male births by year")
plt.show()

### Result

![Marc vs Mark trend](../screenshots/names-trend-chart.png)

**Reading the chart:** *Mark* explodes after 1940, peaks near 59,000 births/year around 1960, then declines steadily. *Marc* follows the same shape at roughly a tenth of the volume, peaking around 1970. Both spellings fade together after 1980 — the trend is about the name itself, not the spelling.

### What this demonstrates
- Connecting Python to SQL Server (`pyodbc` + ODBC Driver 17)
- Writing a filtered, ordered SQL query from Python
- Loading query results into pandas and reshaping with `pivot`
- Producing a readable time-series comparison with matplotlib
- Keeping credentials out of source with environment variables